In [ ]:
from calculate_metrics import *
import torch
import matplotlib.pyplot as plt

## Compare the best and worst mask of a model

In [ ]:
def plot_best_worst_mask(
    run: RunInfo,
    metric: torch.Tensor,
    metric_name: str,
    show_best_other_attempts: bool = False,
):
    image_ids_list = run.image_ids

    # Show the best
    m = metric.max(dim=1)
    max_scores = m.values
    max_value = max_scores.max()
    max_column_indices = m.indices
    # row for image, column for attemps
    max_row_index = max_scores.argmax()
    max_col_index = max_column_indices[max_row_index]
    best_image_id = image_ids_list[max_row_index.item()]
    best_attempt = max_col_index.item()

    image = run.get_original_image(best_image_id)
    mask_ref = run.get_mask_ref(image_id=best_image_id, return_image=True)
    mask_preds = run.get_mask_preds(image_id=best_image_id, return_image=True)
    best_mask_pred = mask_preds[best_attempt]

    if show_best_other_attempts:
        other_mask_preds = [
            mask_preds[i] for i in range(run.attempts) if i != best_attempt
        ]

        fig = plt.figure(figsize=(20, 10))
        gs = fig.add_gridspec(2, 8, wspace=0.1, hspace=0.3)

        ax_orig = fig.add_subplot(gs[0, 1:3])
        ax_orig.imshow(image)
        ax_orig.set_title("Original Image", fontsize=18, fontweight="bold")
        ax_orig.axis("off")

        ax_ref = fig.add_subplot(gs[0, 3:5])
        ax_ref.imshow(mask_ref)
        ax_ref.set_title("Reference Mask", fontsize=18, fontweight="bold")
        ax_ref.axis("off")

        ax_best = fig.add_subplot(gs[0, 5:7])
        ax_best.imshow(best_mask_pred)
        ax_best.set_title(f"Best Prediction ({metric_name} = {max_value:.3f})", fontsize=18, fontweight="bold")
        ax_best.axis("off")

        for idx, mask in enumerate(other_mask_preds):
            ax = fig.add_subplot(gs[1, idx * 2 : (idx + 1) * 2])
            ax.imshow(mask)
            ax.set_title(f"Other Prediction {idx + 1}", fontsize=18, fontweight="bold")
            ax.axis("off")

        plt.tight_layout()
        plt.show()
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(image)
        axes[0].set_title("Original Image", fontsize=18, fontweight="bold")
        axes[0].axis("off")
        axes[1].imshow(mask_ref)
        axes[1].set_title("Reference Mask", fontsize=18, fontweight="bold")
        axes[1].axis("off")
        axes[2].imshow(best_mask_pred)
        axes[2].set_title(f"Best Prediction ({metric_name} = {max_value:.3f})", fontsize=18, fontweight="bold")
        axes[2].axis("off")
        plt.tight_layout()
        plt.show()

    # show the worst with other attempts

    m = metric.min(dim=1)
    min_scores = m.values
    min_value = min_scores.min()
    min_column_indices = m.indices
    # row for image, column for attemps
    min_row_index = min_scores.argmin()
    min_col_index = min_column_indices[min_row_index]
    worst_image_id = image_ids_list[min_row_index.item()]
    worst_attempt = min_col_index.item()

    image = run.get_original_image(worst_image_id)
    mask_ref = run.get_mask_ref(image_id=worst_image_id, return_image=True)
    mask_preds = run.get_mask_preds(image_id=worst_image_id, return_image=True)
    worst_mask_pred = mask_preds[worst_attempt]
    other_mask_preds = [
        mask_preds[i] for i in range(run.attempts) if i != worst_attempt
    ]

    fig = plt.figure(figsize=(20, 10))
    gs = fig.add_gridspec(2, 8, wspace=0.1, hspace=0.3)

    ax_orig = fig.add_subplot(gs[0, 1:3])
    ax_orig.imshow(image)
    ax_orig.set_title("Original Image", fontsize=18, fontweight="bold")
    ax_orig.axis("off")

    ax_ref = fig.add_subplot(gs[0, 3:5])
    ax_ref.imshow(mask_ref)
    ax_ref.set_title("Reference Mask", fontsize=18, fontweight="bold")
    ax_ref.axis("off")

    ax_worst = fig.add_subplot(gs[0, 5:7])
    ax_worst.imshow(worst_mask_pred)
    ax_worst.set_title(f"Worst Prediction ({metric_name} = {min_value:.3f})", fontsize=18, fontweight="bold")
    ax_worst.axis("off")

    for idx, mask in enumerate(other_mask_preds):
        ax = fig.add_subplot(gs[1, idx * 2 : (idx + 1) * 2])
        ax.imshow(mask)
        ax.set_title(f"Other Prediction {idx + 1}", fontsize=18, fontweight="bold")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    return best_image_id, best_attempt, worst_image_id, worst_attempt

In [ ]:
metric_name = "F1"
run = gemini_pro_run_coco

f1, iou, dice = run.load_metrics()
print(
    plot_best_worst_mask(
        run=run, metric=f1, metric_name=metric_name, show_best_other_attempts=True
    )
)

## Compare the best of Gemini Pro Image with Segface on CelebAMask-HQ

In [ ]:
gmnp_f1, _, _ = gemini_pro_run_celeb.load_metrics()
segface_f1, _, _ = segface_run_celeb.load_metrics()

best_idx = segface_f1.flatten().argmax().item()
best_image_id = segface_run_celeb.image_ids[best_idx]

mask_segface = segface_run_celeb.get_mask_preds(
    image_id=best_image_id, return_image=True
)[0]
mask_gemini = gemini_pro_run_celeb.get_mask_preds(
    image_id=best_image_id, return_image=True
)[0]
original_image = gemini_pro_run_celeb.get_original_image(best_image_id)
reference_mask = gemini_pro_run_celeb.get_mask_ref(
    image_id=best_image_id, return_image=True
)

segface_f1_best = segface_f1[best_idx, 0].item()
gemini_pro_f1 = gmnp_f1[best_idx, 0].item()

fig, axs = plt.subplots(2, 2, figsize=(10, 10), gridspec_kw={"hspace": 0.2})

axs[0, 0].imshow(original_image)
axs[0, 0].set_title("Original Image", fontsize=18, fontweight="bold")
axs[0, 0].axis("off")

axs[0, 1].imshow(reference_mask)
axs[0, 1].set_title("Reference Mask", fontsize=18, fontweight="bold")
axs[0, 1].axis("off")

axs[1, 0].imshow(mask_segface)
axs[1, 0].set_title(f"{segface_run_celeb.model_code_name} (F1={segface_f1_best:.4f})", fontsize=18, fontweight="bold")
axs[1, 0].axis("off")

axs[1, 1].imshow(mask_gemini)
axs[1, 1].set_title(f"{gemini_pro_run_celeb.model_code_name} (F1={gemini_pro_f1:.4f})", fontsize=18, fontweight="bold")
axs[1, 1].axis("off")

plt.tight_layout()
plt.show()

## Compare the best of Gemini Pro Image with Oneformer on COCO

In [ ]:
gmnp_f1, _, _ = gemini_pro_run_coco.load_metrics()
gmn_f1, _, _ = gemini_run_coco.load_metrics()
oneformer_f1, _, _ = oneformer_run_coco.load_metrics()

best_idx = oneformer_f1.flatten().argmax().item()
best_image_id = oneformer_run_coco.image_ids[best_idx]

mask_oneformer = oneformer_run_coco.get_mask_preds(
    image_id=best_image_id, return_image=True
)[0]
mask_gemini_pro = gemini_pro_run_coco.get_mask_preds(
    image_id=best_image_id, return_image=True
)[0]
mask_gemini = gemini_run_coco.get_mask_preds(image_id=best_image_id, return_image=True)[
    0
]

original_image = gemini_pro_run_coco.get_original_image(best_image_id)
reference_mask = gemini_pro_run_coco.get_mask_ref(
    image_id=best_image_id, return_image=True
)

oneformer_f1_best = oneformer_f1[best_idx, 0].item()
gemini_pro_f1 = gmnp_f1[best_idx, 0].item()
gemini_f1 = gmn_f1[best_idx, 0].item()

fig, axs = plt.subplots(2, 3, figsize=(10, 10))

axs[0, 0].imshow(original_image)
axs[0, 0].set_title("Original Image")
axs[0, 0].axis("off")

axs[0, 1].imshow(reference_mask)
axs[0, 1].set_title("Reference Mask")
axs[0, 1].axis("off")

axs[0, 2].axis("off")

axs[1, 0].imshow(mask_oneformer)
axs[1, 0].set_title(f"{oneformer_run_coco.model_code_name}: f1={oneformer_f1_best:.4f}")
axs[1, 0].axis("off")

axs[1, 1].imshow(mask_gemini_pro)
axs[1, 1].set_title(f"{gemini_pro_run_coco.model_code_name}: f1={gemini_pro_f1:.4f}")
axs[1, 1].axis("off")

axs[1, 2].imshow(mask_gemini)
axs[1, 2].set_title(f"{gemini_run_coco.model_code_name}: f1={gemini_f1:.4f}")
axs[1, 2].axis("off")

plt.tight_layout()
plt.show()

## Compare results of different models

In [ ]:
runs = [
    gemini_pro_run_celeb,
    gemini_run_celeb,
    gpt_run_celeb,
    sam3_run_celeb,
    segface_run_celeb,
    emu35_run_celeb,
    uni_moe_2_image_run_celeb,
    uni_moe_2_omni_run_celeb,
]

image_id = "0a82f5055de94783bb8e91ed6dfbd547"
original_image = runs[0].get_original_image(image_id)
reference_mask = runs[0].get_mask_ref(image_id=image_id, return_image=True)
attempt_idx = 0

fig = plt.figure(figsize=(15, 20))
gs = fig.add_gridspec(4, 6, hspace=0.2)

ax_orig = fig.add_subplot(gs[0, 1:3])
ax_orig.imshow(original_image)
ax_orig.set_title("Original Image", fontweight="bold", fontsize=18)
ax_orig.axis("off")

ax_ref = fig.add_subplot(gs[0, 3:5])
ax_ref.imshow(reference_mask)
ax_ref.set_title("Reference Mask", fontweight="bold", fontsize=18)
ax_ref.axis("off")

for i, run in enumerate(runs):
    row = (i // 3) + 1
    col_start = (i % 3) * 2
    ax = fig.add_subplot(gs[row, col_start : col_start + 2])
    mask_preds = run.get_mask_preds(image_id=image_id, return_image=True)
    mask = mask_preds[attempt_idx]
    ax.imshow(mask)
    ax.set_title(run.model_code_name, fontweight="bold", fontsize=18)
    ax.axis("off")

plt.tight_layout()
plt.show()